# SensiFake Qwen3-VL Prompt V2 Gold diagnostic

Annotation-quality diagnostic on the frozen 27-image Gold sample. Prompt V2 is evaluated without changing the model, revision, score mapping, or Gold/Silver protocol.

## 1. Imports

Pin the completed pilot's model stack and import PyTorch, evaluation, and checkpointing utilities.

In [ ]:
%pip install -q "transformers==5.16.1" "huggingface_hub==1.29.0" accelerate scikit-learn

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import huggingface_hub
import pandas as pd
import sklearn
import torch
import transformers
from IPython.display import display
from PIL import Image, ImageOps
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
)
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, set_seed

## 2. Globals

The compute configuration and sample are frozen. Prompt V2 is stored verbatim and fingerprinted before any inference.

In [ ]:
SEED = 42
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
MODEL_REVISION = "ebb281ec70b05090aa6165b016eac8ec08e71b17"
MODEL_DTYPE = torch.float16
GPU_COUNT = 2
BATCH_SIZE_PER_GPU = 2
MAX_IMAGE_SIDE = 896
WORKER_START_TIMEOUT_SECONDS = 120
EXPECTED_GOLD_SHA256 = "c783a28509ca200d37d7b9e0f79a4bdd359c3f59f16acf2a0a80759875eb573f"
EXPECTED_GOLD_ROWS = 101
EXPECTED_MANIFEST_ROWS = 600
EXPECTED_DIAGNOSTIC_ROWS = 27
LEVEL_ORDER = ("low", "medium", "high")

KAGGLE_OWNER_ROOT = Path("/kaggle/input/datasets/saracristinabasco")
SOURCE_DATASET_ROOT = KAGGLE_OWNER_ROOT / "sensifake600"
GOLD_DATASET_ROOT = KAGGLE_OWNER_ROOT / "sensifake-gold-development"
GOLD_CSV_PATH = GOLD_DATASET_ROOT / "sensitivity_annotations.csv"
PILOT_ROOT = SOURCE_DATASET_ROOT / "datasets/datasets/openfake/pilot-600"
MANIFEST_PATH = PILOT_ROOT / "manifest.jsonl"
LEGACY_ROOT = SOURCE_DATASET_ROOT / "sensifake-600"

OUTPUT_DIR = Path("/kaggle/working/qwen3vl_prompt_v2_gold_diagnostic")
PREDICTIONS_PATH = OUTPUT_DIR / "prompt_v2_predictions.csv"
METRICS_PATH = OUTPUT_DIR / "prompt_v2_metrics.json"
DISAGREEMENTS_PATH = OUTPUT_DIR / "prompt_v2_disagreements.csv"
RUN_METADATA_PATH = OUTPUT_DIR / "prompt_v2_run_metadata.json"
RESUME_METADATA_PATH = OUTPUT_DIR / "prompt_v2_resume_metadata.json"

V1_PREDICTION_CANDIDATES = (
    Path("/kaggle/working/qwen3vl_gold_pilot/qwen3vl_gold_predictions.csv"),
    Path("data/experiments/openfake/qwen3vl-gold-pilot-v0/qwen3vl_gold_predictions.csv"),
)

GENERATION_PARAMETERS = {
    "max_new_tokens": 220,
    "do_sample": False,
    "use_cache": True,
}

GOLD_REQUIRED_COLUMNS = (
    "content_hash", "blind_id", "public_relevance", "harm_urgency",
    "vulnerability", "sensitivity_score", "sensitivity_level",
    "sensitivity_rationale", "annotation_confidence", "needs_review",
    "annotation_round", "annotated_at",
)
PREDICTION_COLUMNS = (
    "content_hash", "sample_position", "worker_id", "cuda_device",
    "batch_size_per_gpu", "worker_batch_index", "batch_position",
    "batch_image_count", "pred_public_relevance", "pred_harm_urgency",
    "pred_vulnerability", "pred_sensitivity_score", "pred_sensitivity_level",
    "pred_sensitivity_rationale", "pred_annotation_confidence", "parse_ok",
    "parse_error", "raw_model_output", "batch_inference_seconds",
    "batch_seconds_per_image", "completed_at_utc",
)

PROMPT_V2 = """You are annotating the semantic sensitivity of visible image content.

Judge only from information that is visually supported by the image.
Do not determine whether the image is real or synthetic.
Do not infer the dataset source, provenance, generator, model, prompt, caption,
or any other hidden metadata.

The task is NOT to identify the exact person, event, institution, country,
or real-world situation shown in the image.

Instead, assess how sensitive the VISIBLE TYPE OF SCENE could be if the image
were deceptive or misleading.

Use visible contextual cues such as setting, clothing, uniforms, podiums,
stages, crowds, institutional environments, medical equipment, emergency
equipment, military equipment, signs, symbols, or interactions.

Exact identity or exact event recognition is NOT required when the visible
context itself supports a category.

Use this rubric:

public_relevance:

0 = Clearly private, personal, recreational, or ordinary everyday content,
with no visible indication of a public-facing, professional, institutional,
or societally relevant context.

1 = A visibly public-facing, professional, institutional, ceremonial,
organized-event, or otherwise recognizable public context.
The exact person, profession, institution, or event does NOT need to be
identified. Visible contextual evidence is sufficient.

2 = Visible evidence places the scene in a high-stakes societal domain:
politics, elections, government activity, armed conflict, military operations,
disaster or emergency response, or public-health/medical response.
The exact real-world event does NOT need to be known, but the domain must be
supported by visible cues.

IMPORTANT:
Do not default public_relevance to 0 simply because the exact person,
institution, location, or event cannot be identified.


harm_urgency:

0 = If deceptive, the visible content would have little plausible immediate
public consequence beyond ordinary private or benign misunderstanding.

1 = If deceptive, the visible content could plausibly cause reputational harm,
impersonation harm, or general public confusion.
An immediate physical or systemic danger is NOT required for value 1.

2 = If deceptive, the visible content could plausibly affect physical safety,
a democratic process, emergency response, or health-related behavior,
based on the visible type of scene.

public_relevance and harm_urgency are related but distinct.
A scene may be publicly relevant without creating immediate high-stakes harm.


vulnerability:

0 = No visibly vulnerable or distressed subject.

1 = The image visibly depicts patients, injured people, displaced people,
people under threat, or children specifically shown in a crisis, dangerous,
or vulnerable situation.

Do not assign vulnerability=1 merely because a child, elderly person,
or other person is present; visible vulnerability or distress is required.


General decision rule:

Choose the value that best describes the visible scene.
Use the strongest category that is supported by visible evidence, but do not
invent unseen context.

Do not lower a category merely because you cannot identify the exact
real-world person or event.

Assess the potential sensitivity of the depicted TYPE OF CONTENT if it were
used deceptively.


Return exactly one compact JSON object with no Markdown and no extra keys:

{"public_relevance": 0,
 "harm_urgency": 0,
 "vulnerability": 0,
 "sensitivity_rationale": "short rationale based only on visible evidence",
 "annotation_confidence": "high"}

The component values must be integers in their stated ranges.

Do NOT calculate or output the total sensitivity score or final sensitivity
level.

The rationale must be concise (maximum 280 characters) and should mention the
visible cues that motivated the component values.

annotation_confidence must be exactly:
low, medium, or high."""

PROMPT_V2_SHA256 = hashlib.sha256(PROMPT_V2.encode("utf-8")).hexdigest()
EXPECTED_PROMPT_V2_SHA256 = "39fe11682924fd5fc0a20dee94dafb3fcc518fc7b6c28bdd3c38071141673fca"
if PROMPT_V2_SHA256 != EXPECTED_PROMPT_V2_SHA256:
    raise RuntimeError("Prompt V2 text does not match its frozen SHA-256")

RESUME_SIGNATURE = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "prompt_sha256": PROMPT_V2_SHA256,
    "dtype": "torch.float16",
    "max_image_side": MAX_IMAGE_SIDE,
    "gpu_count": GPU_COUNT,
    "model_layout": "one_complete_independent_replica_per_gpu",
    "batch_size_per_gpu": BATCH_SIZE_PER_GPU,
    "generation_parameters": GENERATION_PARAMETERS,
}

V1_REFERENCE_101 = {
    "sample_size": 101,
    "human_distribution": {"low": 54, "medium": 38, "high": 9},
    "qwen_v1_distribution": {"low": 100, "medium": 1, "high": 0},
    "sensitivity_level_accuracy": 0.5347,
    "macro_f1": 0.2338,
    "linear_weighted_kappa": 0.0191,
    "quadratic_weighted_kappa": 0.0391,
    "exact_score_accuracy": 0.2871,
    "score_mae": 1.0693,
    "high_recall": 0.0,
    "component_accuracy": {
        "public_relevance": 0.3762,
        "harm_urgency": 0.9109,
        "vulnerability": 0.7921,
    },
    "comparison_warning": (
        "Context only: these aggregate V1 results use 101 images and are not "
        "a statistically valid direct comparison with the 27-image V2 diagnostic."
    ),
}

random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
NOTEBOOK_STARTED_PERF = time.perf_counter()

## 3. Utils

Utilities preserve the original strict parser, enforce model blinding, perform true batch-two inference, and atomically checkpoint every completed batch.

In [ ]:
@dataclass(frozen=True)
class DiagnosticItem:
    sample_position: int
    content_hash: str
    image: Image.Image


@dataclass
class ModelReplica:
    worker_id: int
    device: torch.device
    processor: Any
    model: Any
    effective_device_map: dict[str, str]
    hf_device_map_exposed: bool
    validation_report: dict[str, Any]


def utc_now() -> str:
    return datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def seeded_order_key(content_hash: str, purpose: str) -> str:
    value = f"{SEED}:{purpose}:{content_hash}".encode()
    return hashlib.sha256(value).hexdigest()


def derived_level(score: int) -> str:
    if score not in range(6):
        raise ValueError(f"sensitivity score outside 0..5: {score}")
    if score <= 1:
        return "low"
    if score <= 3:
        return "medium"
    return "high"


def reject_duplicate_json_keys(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    result: dict[str, Any] = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate JSON key: {key}")
        result[key] = value
    return result


def parse_model_json(raw_output: str) -> dict[str, Any]:
    expected = {
        "public_relevance", "harm_urgency", "vulnerability",
        "sensitivity_rationale", "annotation_confidence",
    }
    payload = json.loads(
        raw_output.strip(), object_pairs_hook=reject_duplicate_json_keys
    )
    if not isinstance(payload, dict):
        raise TypeError("response must be one JSON object")
    if set(payload) != expected:
        missing = sorted(expected - set(payload))
        extra = sorted(set(payload) - expected)
        raise ValueError(
            f"JSON keys do not match schema; missing={missing}, extra={extra}"
        )

    ranges = {
        "public_relevance": range(3),
        "harm_urgency": range(3),
        "vulnerability": range(2),
    }
    for field, allowed in ranges.items():
        value = payload[field]
        if type(value) is not int or value not in allowed:
            raise ValueError(f"{field} must be an integer in {list(allowed)}")

    rationale = payload["sensitivity_rationale"]
    if not isinstance(rationale, str) or not rationale.strip():
        raise ValueError("sensitivity_rationale must be a non-empty string")
    if len(rationale.strip()) > 280:
        raise ValueError("sensitivity_rationale exceeds 280 characters")
    confidence = payload["annotation_confidence"]
    if confidence not in {"low", "medium", "high"}:
        raise ValueError("annotation_confidence must be low, medium, or high")

    payload["sensitivity_rationale"] = rationale.strip()
    score = (
        payload["public_relevance"]
        + payload["harm_urgency"]
        + payload["vulnerability"]
    )
    payload["sensitivity_score"] = score
    payload["sensitivity_level"] = derived_level(score)
    return payload


def atomic_write_csv(frame: pd.DataFrame, path: Path) -> None:
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def json_compatible(value: Any) -> Any:
    if isinstance(value, dict):
        return {
            str(key): json_compatible(item) for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [json_compatible(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if hasattr(value, "item") and callable(value.item):
        value = value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def atomic_write_json(payload: dict[str, Any], path: Path) -> None:
    temporary = path.with_name(path.name + ".tmp")
    serialized = json.dumps(
        json_compatible(payload),
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )
    temporary.write_text(serialized + "\n", encoding="utf-8")
    os.replace(temporary, path)

def load_image_in_memory(path: Path) -> Image.Image:
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(source).convert("RGB")
        if max(image.size) > MAX_IMAGE_SIDE:
            image.thumbnail(
                (MAX_IMAGE_SIDE, MAX_IMAGE_SIDE), Image.Resampling.LANCZOS
            )
        return image.copy()


def blind_messages(image: Image.Image) -> list[dict[str, Any]]:
    # This is the complete model-facing boundary: pixels plus Prompt V2 only.
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT_V2},
            ],
        }
    ]


def item_batches(
    items: list[DiagnosticItem], batch_size: int
) -> list[list[DiagnosticItem]]:
    return [
        items[start : start + batch_size]
        for start in range(0, len(items), batch_size)
    ]


def normalize_device(value: Any) -> str:
    if isinstance(value, int):
        return f"cuda:{value}"
    text = str(value)
    if text.isdigit():
        return f"cuda:{text}"
    if text == "cuda":
        return "cuda:0"
    return text


def verify_complete_replica(replica: ModelReplica) -> dict[str, Any]:
    intended = str(replica.device)
    parameter_devices = {
        str(parameter.device) for parameter in replica.model.parameters()
    }
    if parameter_devices != {intended}:
        raise RuntimeError(
            f"replica {replica.worker_id} parameters are not wholly on "
            f"{intended}: {sorted(parameter_devices)}"
        )

    parameter_dtypes = {
        str(parameter.dtype) for parameter in replica.model.parameters()
    }
    floating_parameter_dtypes = {
        parameter.dtype
        for parameter in replica.model.parameters()
        if parameter.is_floating_point()
    }
    if floating_parameter_dtypes != {torch.float16}:
        raise RuntimeError(
            f"replica {replica.worker_id} floating parameter dtypes are not "
            f"FP16: {floating_parameter_dtypes}"
        )
    if getattr(replica.model, "is_quantized", False):
        raise RuntimeError("quantization is forbidden")
    if getattr(replica.model, "quantization_method", None) is not None:
        raise RuntimeError("a quantization method was detected")

    nonpersistent_names: set[str] = set()
    for module_name, module in replica.model.named_modules():
        for buffer_name in getattr(
            module, "_non_persistent_buffers_set", set()
        ):
            qualified = (
                f"{module_name}.{buffer_name}"
                if module_name
                else buffer_name
            )
            nonpersistent_names.add(qualified)

    buffer_devices: set[str] = set()
    buffer_dtypes: set[str] = set()
    harmless_off_device_buffers: list[dict[str, Any]] = []
    forbidden_off_device_buffers: list[dict[str, Any]] = []
    for buffer_name, buffer in replica.model.named_buffers():
        buffer_device = str(buffer.device)
        buffer_devices.add(buffer_device)
        buffer_dtypes.add(str(buffer.dtype))
        if buffer_device == intended:
            continue
        is_nonpersistent = buffer_name in nonpersistent_names
        is_harmless_nonpersistent_cpu_buffer = (
            buffer.device.type == "cpu"
            and is_nonpersistent
        )
        details = {
            "name": buffer_name,
            "device": buffer_device,
            "dtype": str(buffer.dtype),
            "numel": int(buffer.numel()),
            "nonpersistent": is_nonpersistent,
        }
        if is_harmless_nonpersistent_cpu_buffer:
            harmless_off_device_buffers.append(details)
        else:
            forbidden_off_device_buffers.append(details)
    if forbidden_off_device_buffers:
        raise RuntimeError(
            f"replica {replica.worker_id} has persistent, meta, or otherwise "
            f"unsafe buffers off {intended}: "
            f"{forbidden_off_device_buffers[:10]}"
        )

    for module in replica.model.modules():
        hook = getattr(module, "_hf_hook", None)
        if hook is not None and bool(getattr(hook, "offload", False)):
            raise RuntimeError(
                f"replica {replica.worker_id} has an active offload hook"
            )

    if replica.hf_device_map_exposed:
        mapped_devices = set(replica.effective_device_map.values())
        if mapped_devices != {intended}:
            raise RuntimeError(
                f"replica {replica.worker_id} exposed device map is not "
                f"confined to {intended}: {sorted(mapped_devices)}"
            )

    gib = 1024 ** 3
    return {
        "worker_id": replica.worker_id,
        "device": intended,
        "parameter_devices": sorted(parameter_devices),
        "buffer_devices": sorted(buffer_devices),
        "parameter_dtypes": sorted(parameter_dtypes),
        "buffer_dtypes": sorted(buffer_dtypes),
        "harmless_nonpersistent_off_device_buffers": (
            harmless_off_device_buffers
        ),
        "allocated_vram_gib": (
            torch.cuda.memory_allocated(replica.device) / gib
        ),
        "reserved_vram_gib": (
            torch.cuda.memory_reserved(replica.device) / gib
        ),
        "hf_device_map": (
            replica.effective_device_map
            if replica.hf_device_map_exposed
            else "not exposed"
        ),
        "reported_device_map_for_metadata": replica.effective_device_map,
        "quantized": False,
        "offload_hooks": False,
    }

def load_replica(worker_id: int) -> ModelReplica:
    device = torch.device(f"cuda:{worker_id}")
    torch.cuda.set_device(device)
    processor = AutoProcessor.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False
    )
    processor.tokenizer.padding_side = "left"
    if processor.tokenizer.pad_token_id is None:
        if processor.tokenizer.eos_token_id is None:
            raise RuntimeError(
                "Qwen tokenizer has neither pad_token_id nor eos_token_id; "
                "true batched generation cannot proceed."
            )
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    try:
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            revision=MODEL_REVISION,
            dtype=MODEL_DTYPE,
            device_map={"": str(device)},
            attn_implementation="sdpa",
            low_cpu_mem_usage=True,
            trust_remote_code=False,
        )
    except (torch.OutOfMemoryError, RuntimeError) as exc:
        is_oom = isinstance(exc, torch.OutOfMemoryError) or (
            "out of memory" in str(exc).lower()
        )
        if not is_oom:
            raise
        torch.cuda.empty_cache()
        raise RuntimeError(
            "A complete unquantized FP16 replica could not be loaded on "
            f"{device}. GPU memory is already occupied or the session is "
            "fragmented. Restart or factory-reset the Kaggle session and "
            "rerun from the top. No sharding or quantization fallback was "
            "attempted."
        ) from exc

    model.eval()
    raw_map = getattr(model, "hf_device_map", None)
    map_exposed = isinstance(raw_map, dict) and bool(raw_map)
    effective_map = (
        {
            str(module): normalize_device(mapped_device)
            for module, mapped_device in raw_map.items()
        }
        if map_exposed
        else {"": str(device)}
    )
    replica = ModelReplica(
        worker_id=worker_id,
        device=device,
        processor=processor,
        model=model,
        effective_device_map=effective_map,
        hf_device_map_exposed=map_exposed,
        validation_report={},
    )
    replica.validation_report = verify_complete_replica(replica)
    print(
        "Replica placement validation:\n"
        + json.dumps(
            json_compatible(replica.validation_report),
            indent=2,
            sort_keys=True,
            allow_nan=False,
        )
    )
    return replica

def load_resume_checkpoint(
    expected_positions: dict[str, int],
) -> pd.DataFrame:
    allowed_hashes = set(expected_positions)
    predictions_exist = PREDICTIONS_PATH.is_file()
    metadata_exist = RESUME_METADATA_PATH.is_file()
    if predictions_exist and not metadata_exist:
        raise RuntimeError(
            f"Prediction checkpoint {PREDICTIONS_PATH} exists without "
            f"{RESUME_METADATA_PATH}. Do not delete the predictions. Restore "
            "or inspect the matching resume metadata before rerunning."
        )

    if metadata_exist:
        try:
            saved_signature = json.loads(
                RESUME_METADATA_PATH.read_text(encoding="utf-8")
            )
        except (json.JSONDecodeError, UnicodeDecodeError) as exc:
            if predictions_exist:
                recovery = (
                    f"A prediction checkpoint also exists at "
                    f"{PREDICTIONS_PATH}; do not delete either file "
                    "automatically. Inspect and recover the metadata before "
                    "resuming."
                )
            else:
                recovery = (
                    "No prediction checkpoint exists, so the documented safe "
                    f"cleanup is to delete only {RESUME_METADATA_PATH}, then "
                    "rerun from the preflight cell."
                )
            raise RuntimeError(
                f"Malformed resume metadata JSON at "
                f"{RESUME_METADATA_PATH}: {exc}. {recovery}"
            ) from exc
        if not isinstance(saved_signature, dict):
            raise RuntimeError(
                f"Resume metadata {RESUME_METADATA_PATH} must contain one "
                "JSON object."
            )
        if saved_signature != RESUME_SIGNATURE:
            raise RuntimeError(
                "Incompatible checkpoint metadata. Refusing to mix "
                "predictions from a different model, revision, prompt, "
                "dtype, image size, or batch configuration."
            )
    else:
        atomic_write_json(RESUME_SIGNATURE, RESUME_METADATA_PATH)

    if not predictions_exist:
        return pd.DataFrame(columns=PREDICTION_COLUMNS)

    try:
        checkpoint = pd.read_csv(
            PREDICTIONS_PATH, keep_default_na=False
        )
    except (
        pd.errors.EmptyDataError,
        pd.errors.ParserError,
        UnicodeDecodeError,
    ) as exc:
        raise RuntimeError(
            f"Prediction checkpoint {PREDICTIONS_PATH} is unreadable: {exc}. "
            "Do not delete it automatically; inspect or recover the file."
        ) from exc
    if tuple(checkpoint.columns) != PREDICTION_COLUMNS:
        raise RuntimeError(
            f"Prediction checkpoint {PREDICTIONS_PATH} has an incompatible "
            "schema."
        )
    if len(checkpoint) > EXPECTED_DIAGNOSTIC_ROWS:
        raise RuntimeError("Prediction checkpoint contains more than 27 rows")
    if checkpoint["content_hash"].duplicated().any():
        raise RuntimeError("Prediction checkpoint contains duplicate hashes")

    checkpoint["content_hash"] = checkpoint["content_hash"].astype(str)
    unknown = set(checkpoint["content_hash"]) - allowed_hashes
    if unknown:
        raise RuntimeError(
            "Prediction checkpoint contains hashes outside this sample: "
            f"{sorted(unknown)}"
        )

    numeric_positions = pd.to_numeric(
        checkpoint["sample_position"], errors="coerce"
    )
    if numeric_positions.isna().any() or (
        numeric_positions % 1 != 0
    ).any():
        raise RuntimeError(
            "Prediction checkpoint contains invalid sample_position values"
        )
    checkpoint["sample_position"] = numeric_positions.astype(int)
    wrong_positions = {
        row.content_hash: int(row.sample_position)
        for row in checkpoint[
            ["content_hash", "sample_position"]
        ].itertuples(index=False)
        if expected_positions[row.content_hash] != int(row.sample_position)
    }
    if wrong_positions:
        raise RuntimeError(
            "Prediction checkpoint sample positions do not match the frozen "
            f"27-image ordering: {wrong_positions}"
        )

    normalized_parse = checkpoint["parse_ok"].map(
        lambda value: {
            "true": True,
            "false": False,
        }.get(str(value).strip().lower())
    )
    if normalized_parse.isna().any():
        raise RuntimeError(
            "Prediction checkpoint contains invalid parse_ok values"
        )
    checkpoint["parse_ok"] = normalized_parse.astype(bool)

    worker_ids = pd.to_numeric(
        checkpoint["worker_id"], errors="coerce"
    )
    batch_sizes = pd.to_numeric(
        checkpoint["batch_size_per_gpu"], errors="coerce"
    )
    if worker_ids.isna().any() or not worker_ids.isin(range(GPU_COUNT)).all():
        raise RuntimeError("Prediction checkpoint has invalid worker IDs")
    if (
        batch_sizes.isna().any()
        or not (batch_sizes == BATCH_SIZE_PER_GPU).all()
    ):
        raise RuntimeError(
            "Prediction checkpoint does not use frozen batch size 2"
        )
    expected_cuda_devices = worker_ids.astype(int).map(
        lambda worker_id: f"cuda:{worker_id}"
    )
    if not (
        checkpoint["cuda_device"].astype(str) == expected_cuda_devices
    ).all():
        raise RuntimeError(
            "Prediction checkpoint worker/GPU metadata is inconsistent"
        )

    parsed_rows = checkpoint["parse_ok"]
    numeric_prediction_columns = [
        "pred_public_relevance",
        "pred_harm_urgency",
        "pred_vulnerability",
        "pred_sensitivity_score",
    ]
    parsed_numeric = checkpoint.loc[
        parsed_rows, numeric_prediction_columns
    ].apply(pd.to_numeric, errors="coerce")
    if parsed_numeric.isna().any().any():
        raise RuntimeError(
            "Strictly parsed checkpoint rows contain missing/non-numeric "
            "component or score values"
        )
    allowed_component_values = {
        "pred_public_relevance": range(3),
        "pred_harm_urgency": range(3),
        "pred_vulnerability": range(2),
    }
    for column, allowed in allowed_component_values.items():
        values = parsed_numeric[column]
        if not (
            (values % 1 == 0) & values.isin(list(allowed))
        ).all():
            raise RuntimeError(
                f"Checkpoint {column} values violate the frozen rubric"
            )
    rationales = checkpoint.loc[
        parsed_rows, "pred_sensitivity_rationale"
    ].astype(str)
    if (
        (rationales.str.strip().str.len() == 0).any()
        or (rationales.str.strip().str.len() > 280).any()
    ):
        raise RuntimeError(
            "Strictly parsed checkpoint rationales are empty or too long"
        )
    confidences = checkpoint.loc[
        parsed_rows, "pred_annotation_confidence"
    ].astype(str)
    if not confidences.isin(["low", "medium", "high"]).all():
        raise RuntimeError(
            "Strictly parsed checkpoint confidence values are invalid"
        )
    if not parsed_numeric.empty:
        expected_scores = (
            parsed_numeric["pred_public_relevance"]
            + parsed_numeric["pred_harm_urgency"]
            + parsed_numeric["pred_vulnerability"]
        )
        if not (
            expected_scores == parsed_numeric["pred_sensitivity_score"]
        ).all():
            raise RuntimeError(
                "Checkpoint sensitivity scores are inconsistent with "
                "component sums"
            )
        expected_levels = expected_scores.astype(int).map(derived_level)
        if not (
            expected_levels.to_numpy()
            == checkpoint.loc[
                parsed_rows, "pred_sensitivity_level"
            ].astype(str).to_numpy()
        ).all():
            raise RuntimeError(
                "Checkpoint sensitivity levels are inconsistent with scores"
            )
    return checkpoint

def checkpoint_completed_batch(records: list[dict[str, Any]]) -> None:
    with CHECKPOINT_LOCK:
        incoming_hashes = {
            str(record["content_hash"]) for record in records
        }
        if len(incoming_hashes) != len(records):
            raise RuntimeError(
                "completed batch contains duplicate content_hash values"
            )
        overlap = incoming_hashes & set(CHECKPOINT_RECORDS)
        if overlap:
            raise RuntimeError(
                "refusing to overwrite completed predictions: "
                f"{sorted(overlap)}"
            )
        wrong_positions = {
            str(record["content_hash"]): int(record["sample_position"])
            for record in records
            if EXPECTED_SAMPLE_POSITIONS[str(record["content_hash"])]
            != int(record["sample_position"])
        }
        if wrong_positions:
            raise RuntimeError(
                "completed batch has incorrect frozen sample positions: "
                f"{wrong_positions}"
            )

        candidate_records = dict(CHECKPOINT_RECORDS)
        for record in records:
            candidate_records[str(record["content_hash"])] = record
        ordered = sorted(
            candidate_records.values(),
            key=lambda record: int(record["sample_position"]),
        )
        atomic_write_csv(
            pd.DataFrame(ordered, columns=PREDICTION_COLUMNS),
            PREDICTIONS_PATH,
        )
        CHECKPOINT_RECORDS.clear()
        CHECKPOINT_RECORDS.update(candidate_records)

def infer_batch(
    replica: ModelReplica,
    items: list[DiagnosticItem],
    worker_batch_index: int,
) -> list[dict[str, Any]]:
    if not 1 <= len(items) <= BATCH_SIZE_PER_GPU:
        raise RuntimeError(
            f"invalid worker batch size: {len(items)}"
        )
    torch.cuda.set_device(replica.device)
    conversations = [blind_messages(item.image) for item in items]
    torch.cuda.synchronize(replica.device)
    batch_started = time.perf_counter()
    inputs = replica.processor.apply_chat_template(
        conversations,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        padding=True,
    )
    inputs.pop("token_type_ids", None)
    if inputs["input_ids"].shape[0] != len(items):
        raise RuntimeError(
            "AutoProcessor did not create one sequence per conversation"
        )
    input_length = int(inputs["input_ids"].shape[-1])
    inputs = inputs.to(replica.device)
    with torch.inference_mode():
        generated = replica.model.generate(
            **inputs, **GENERATION_PARAMETERS
        )
    torch.cuda.synchronize(replica.device)
    batch_seconds = time.perf_counter() - batch_started
    if (
        generated.ndim != 2
        or generated.shape[0] != len(items)
        or generated.shape[1] < input_length
    ):
        raise RuntimeError(
            "generate() returned an unexpected batched sequence shape: "
            f"{tuple(generated.shape)}"
        )

    # Left-padded conversations share the processor's padded input width.
    # generate() prepends that full width to every returned sequence.
    generated_only = generated[:, input_length:]
    raw_outputs = replica.processor.batch_decode(
        generated_only,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    if len(raw_outputs) != len(items):
        raise RuntimeError(
            "batch decoding did not return one response per image"
        )

    records: list[dict[str, Any]] = []
    for batch_position, (item, raw_output) in enumerate(
        zip(items, raw_outputs, strict=True)
    ):
        raw_output = raw_output.strip()
        try:
            parsed = parse_model_json(raw_output)
            parse_ok = True
            parse_error = ""
        except (json.JSONDecodeError, TypeError, ValueError) as exc:
            parsed = None
            parse_ok = False
            parse_error = f"{type(exc).__name__}: {exc}"
        records.append(
            {
                "content_hash": item.content_hash,
                "sample_position": item.sample_position,
                "worker_id": replica.worker_id,
                "cuda_device": str(replica.device),
                "batch_size_per_gpu": BATCH_SIZE_PER_GPU,
                "worker_batch_index": worker_batch_index,
                "batch_position": batch_position,
                "batch_image_count": len(items),
                "pred_public_relevance": (
                    parsed["public_relevance"] if parsed else None
                ),
                "pred_harm_urgency": (
                    parsed["harm_urgency"] if parsed else None
                ),
                "pred_vulnerability": (
                    parsed["vulnerability"] if parsed else None
                ),
                "pred_sensitivity_score": (
                    parsed["sensitivity_score"] if parsed else None
                ),
                "pred_sensitivity_level": (
                    parsed["sensitivity_level"] if parsed else None
                ),
                "pred_sensitivity_rationale": (
                    parsed["sensitivity_rationale"] if parsed else None
                ),
                "pred_annotation_confidence": (
                    parsed["annotation_confidence"] if parsed else None
                ),
                "parse_ok": parse_ok,
                "parse_error": parse_error,
                "raw_model_output": raw_output,
                "batch_inference_seconds": round(batch_seconds, 6),
                "batch_seconds_per_image": round(
                    batch_seconds / len(items), 6
                ),
                "completed_at_utc": utc_now(),
            }
        )
    del inputs, generated, generated_only, conversations, raw_outputs
    checkpoint_completed_batch(records)
    return records

def worker_run(
    replica: ModelReplica,
    assigned_items: list[DiagnosticItem],
    start_event: threading.Event,
) -> list[dict[str, Any]]:
    torch.cuda.set_device(replica.device)
    torch.cuda.synchronize(replica.device)
    if not start_event.wait(timeout=WORKER_START_TIMEOUT_SECONDS):
        raise RuntimeError(
            f"worker {replica.worker_id} timed out waiting for inference start"
        )
    records: list[dict[str, Any]] = []
    for batch_index, batch in enumerate(
        item_batches(assigned_items, BATCH_SIZE_PER_GPU)
    ):
        records.extend(infer_batch(replica, batch, batch_index))
    return records

## 4. Data

Validate the frozen Gold artifact and canonical manifest, then reproduce the benchmark's SHA-256 sample: 9 LOW, 9 MEDIUM, and all 9 HIGH. Human annotations remain separate until evaluation.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not GOLD_CSV_PATH.is_file():
    raise FileNotFoundError(f"Gold annotations not found: {GOLD_CSV_PATH}")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"canonical pilot manifest not found: {MANIFEST_PATH}")
if sha256_file(GOLD_CSV_PATH) != EXPECTED_GOLD_SHA256:
    raise RuntimeError("Gold CSV SHA-256 does not match the frozen artifact")

gold_frame = pd.read_csv(GOLD_CSV_PATH, keep_default_na=False)
if tuple(gold_frame.columns) != GOLD_REQUIRED_COLUMNS:
    raise RuntimeError("Gold CSV columns do not exactly match the frozen schema")
if len(gold_frame) != EXPECTED_GOLD_ROWS:
    raise RuntimeError(f"expected 101 Gold rows, found {len(gold_frame)}")
if gold_frame["content_hash"].duplicated().any():
    raise RuntimeError("Gold content_hash values are not unique")
gold_distribution = (
    gold_frame["sensitivity_level"]
    .value_counts()
    .reindex(LEVEL_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)
if gold_distribution != {"low": 54, "medium": 38, "high": 9}:
    raise RuntimeError(f"unexpected frozen Gold distribution: {gold_distribution}")

selected_rows: list[dict[str, str]] = []
for level in LEVEL_ORDER:
    candidates = gold_frame.loc[
        gold_frame["sensitivity_level"] == level,
        ["content_hash", "sensitivity_level"],
    ].to_dict("records")
    if len(candidates) < 9:
        raise RuntimeError(f"not enough Gold {level.upper()} cases")
    ordered = sorted(
        candidates,
        key=lambda row: seeded_order_key(
            str(row["content_hash"]), f"select-{level}"
        ),
    )
    selected_rows.extend(ordered[:9])
selected_rows = sorted(
    selected_rows,
    key=lambda row: seeded_order_key(
        str(row["content_hash"]), "benchmark-order"
    ),
)
selection_frame = pd.DataFrame(selected_rows)
selection_frame["sample_position"] = range(len(selection_frame))
sample_distribution = (
    selection_frame["sensitivity_level"]
    .value_counts()
    .reindex(LEVEL_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)
if sample_distribution != {"low": 9, "medium": 9, "high": 9}:
    raise RuntimeError(f"diagnostic sample is not 9/9/9: {sample_distribution}")
all_high_hashes = set(
    gold_frame.loc[
        gold_frame["sensitivity_level"] == "high", "content_hash"
    ].astype(str)
)
selected_high_hashes = set(
    selection_frame.loc[
        selection_frame["sensitivity_level"] == "high", "content_hash"
    ].astype(str)
)
if selected_high_hashes != all_high_hashes:
    raise RuntimeError("diagnostic sample does not contain all 9 Gold HIGH cases")
if (
    len(selection_frame) != EXPECTED_DIAGNOSTIC_ROWS
    or selection_frame["content_hash"].duplicated().any()
):
    raise RuntimeError("diagnostic sample must contain 27 unique hashes")

gold_evaluation = selection_frame[["content_hash", "sample_position"]].merge(
    gold_frame,
    on="content_hash",
    how="left",
    validate="one_to_one",
)

manifest_records: list[dict[str, Any]] = []
with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        if not line.strip():
            raise RuntimeError(f"blank manifest line at {line_number}")
        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            raise RuntimeError(
                f"invalid manifest JSON at line {line_number}"
            ) from exc
        if not isinstance(record, dict):
            raise TypeError(f"manifest line {line_number} is not an object")
        manifest_records.append(record)
if len(manifest_records) != EXPECTED_MANIFEST_ROWS:
    raise RuntimeError(
        f"expected 600 manifest rows, found {len(manifest_records)}"
    )
manifest_frame = pd.DataFrame(manifest_records)
if not {"content_hash", "relative_image_path"}.issubset(
    manifest_frame.columns
):
    raise RuntimeError("canonical manifest lacks path-resolution fields")

selected_hashes = set(selection_frame["content_hash"].astype(str))
manifest_matches = manifest_frame[
    manifest_frame["content_hash"].isin(selected_hashes)
][["content_hash", "relative_image_path"]].copy()
match_counts = manifest_matches["content_hash"].value_counts()
bad_counts = {
    content_hash: int(match_counts.get(content_hash, 0))
    for content_hash in selected_hashes
    if match_counts.get(content_hash, 0) != 1
}
if bad_counts:
    raise RuntimeError(
        f"selected hashes do not resolve exactly once: {bad_counts}"
    )

resolved = selection_frame[["content_hash", "sample_position"]].merge(
    manifest_matches,
    on="content_hash",
    how="left",
    validate="one_to_one",
)
pilot_resolved = PILOT_ROOT.resolve()
legacy_resolved = LEGACY_ROOT.resolve(strict=False)
resolved_paths: set[Path] = set()
resolved_rows: list[dict[str, Any]] = []
for row in resolved.itertuples(index=False):
    image_path = (PILOT_ROOT / str(row.relative_image_path)).resolve()
    if not image_path.is_relative_to(pilot_resolved):
        raise RuntimeError(
            f"resolved path escapes canonical pilot root: {image_path}"
        )
    if image_path.is_relative_to(legacy_resolved):
        raise RuntimeError(f"legacy duplicate path is forbidden: {image_path}")
    if not image_path.is_file():
        raise FileNotFoundError(f"diagnostic image is missing: {image_path}")
    if image_path in resolved_paths:
        raise RuntimeError(f"duplicate resolved image path: {image_path}")
    resolved_paths.add(image_path)
    resolved_rows.append(
        {
            "content_hash": str(row.content_hash),
            "sample_position": int(row.sample_position),
            "image_path": image_path,
        }
    )
if len(resolved_rows) != EXPECTED_DIAGNOSTIC_ROWS:
    raise RuntimeError("exactly 27 canonical image paths must resolve")

EXPECTED_SAMPLE_POSITIONS = {
    str(row.content_hash): int(row.sample_position)
    for row in selection_frame[
        ["content_hash", "sample_position"]
    ].itertuples(index=False)
}
checkpoint_frame = load_resume_checkpoint(EXPECTED_SAMPLE_POSITIONS)
CHECKPOINT_RECORDS = {
    str(record["content_hash"]): record
    for record in checkpoint_frame.to_dict("records")
}
CHECKPOINT_LOCK = threading.Lock()
initial_resume_count = len(CHECKPOINT_RECORDS)
pending_hashes = selected_hashes - set(CHECKPOINT_RECORDS)
inference_items = tuple(
    DiagnosticItem(
        sample_position=row["sample_position"],
        content_hash=row["content_hash"],
        image=load_image_in_memory(Path(row["image_path"])),
    )
    for row in resolved_rows
    if row["content_hash"] in pending_hashes
)

# Enforce the boundary: model-facing objects contain no label, score, path,
# blind_id, provenance, caption, source, or real/fake information.
del (
    gold_frame,
    selected_rows,
    selection_frame,
    manifest_records,
    manifest_frame,
    manifest_matches,
    resolved,
    resolved_paths,
    resolved_rows,
    checkpoint_frame,
    pending_hashes,
)
print(
    f"Validated frozen data and 27-image 9/9/9 sample; "
    f"resume={initial_resume_count}, pending={len(inference_items)}."
)

## 5. Network

Load one complete independent unquantized FP16 replica on each T4. Any loading OOM stops with an explicit Kaggle restart/factory-reset instruction; sharding is never attempted.

In [ ]:
if transformers.__version__ != "5.16.1":
    raise RuntimeError(
        f"unexpected Transformers version: {transformers.__version__}; "
        "rerun the pinned Imports cell and restart the kernel if needed."
    )
if huggingface_hub.__version__ != "1.29.0":
    raise RuntimeError(
        f"unexpected huggingface_hub version: "
        f"{huggingface_hub.__version__}; rerun the pinned Imports cell "
        "and restart the kernel if needed."
    )
if not torch.cuda.is_available() or torch.cuda.device_count() != GPU_COUNT:
    raise RuntimeError(
        f"this diagnostic requires exactly two CUDA GPUs, "
        f"found {torch.cuda.device_count()}"
    )
GPU_NAMES = [
    torch.cuda.get_device_name(index) for index in range(GPU_COUNT)
]
if not all("T4" in name.upper() for name in GPU_NAMES):
    raise RuntimeError(
        f"this diagnostic is frozen for Tesla T4 x2, found {GPU_NAMES}"
    )
torch.cuda.manual_seed_all(SEED)

gib = 1024 ** 3
PREFLIGHT_REPORT = {
    "status": "passed",
    "gold_csv": str(GOLD_CSV_PATH),
    "gold_sha256": EXPECTED_GOLD_SHA256,
    "gold_rows": EXPECTED_GOLD_ROWS,
    "manifest": str(MANIFEST_PATH),
    "manifest_rows": EXPECTED_MANIFEST_ROWS,
    "resolved_diagnostic_hashes": len(EXPECTED_SAMPLE_POSITIONS),
    "sample_distribution": sample_distribution,
    "prompt_v2_sha256": PROMPT_V2_SHA256,
    "resume_metadata": str(RESUME_METADATA_PATH),
    "resume_metadata_valid_json": True,
    "checkpoint_predictions_found": initial_resume_count,
    "pending_predictions": len(inference_items),
    "gpu_names": GPU_NAMES,
    "preload_gpu_memory": {
        f"cuda:{index}": {
            "allocated_gib": torch.cuda.memory_allocated(index) / gib,
            "reserved_gib": torch.cuda.memory_reserved(index) / gib,
        }
        for index in range(GPU_COUNT)
    },
    "versions": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "huggingface_hub": huggingface_hub.__version__,
        "sklearn": sklearn.__version__,
        "cuda": torch.version.cuda,
    },
}
print(
    "Self-diagnostic preflight passed before model loading:\n"
    + json.dumps(
        json_compatible(PREFLIGHT_REPORT),
        indent=2,
        sort_keys=True,
        allow_nan=False,
    )
)

if inference_items:
    replicas = []
    for worker_id in range(GPU_COUNT):
        replicas.append(load_replica(worker_id))
    if replicas[0].model is replicas[1].model:
        raise RuntimeError("GPU workers unexpectedly share one model object")
else:
    replicas = []
    print("All 27 hashes are checkpointed; model loading is skipped.")

REPLICA_DEVICE_MAPS = [
    {
        "worker_id": replica.worker_id,
        "device": str(replica.device),
        "hf_device_map_exposed": replica.hf_device_map_exposed,
        "hf_device_map": (
            replica.effective_device_map
            if replica.hf_device_map_exposed
            else "not exposed"
        ),
        "reported_device_map_for_metadata": (
            replica.effective_device_map
        ),
        "placement_validation": replica.validation_report,
    }
    for replica in replicas
]
display(pd.DataFrame(REPLICA_DEVICE_MAPS))

## 6. Train / Inference

There is NO training: both replicas remain in eval mode and their weights are never updated. Two worker threads run true batch-two multimodal inference concurrently. Each completed batch is atomically checkpointed.

In [ ]:
new_attempt_count = 0
inference_wall_seconds = 0.0

if inference_items:
    assignments = [
        [
            item
            for item in inference_items
            if item.sample_position % GPU_COUNT == worker_id
        ]
        for worker_id in range(GPU_COUNT)
    ]
    assigned_hashes = [
        item.content_hash
        for assignment in assignments
        for item in assignment
    ]
    if (
        len(assigned_hashes) != len(set(assigned_hashes))
        or set(assigned_hashes)
        != {item.content_hash for item in inference_items}
    ):
        raise RuntimeError(
            "dual-GPU pending-work split is incomplete or duplicated"
        )

    active_replicas = [
        replica
        for replica in replicas
        if assignments[replica.worker_id]
    ]
    if not active_replicas:
        raise RuntimeError(
            "pending images exist but no GPU worker received work"
        )
    start_event = threading.Event()
    for replica in active_replicas:
        torch.cuda.synchronize(replica.device)

    try:
        with ThreadPoolExecutor(
            max_workers=len(active_replicas)
        ) as executor:
            futures = [
                executor.submit(
                    worker_run,
                    replica,
                    assignments[replica.worker_id],
                    start_event,
                )
                for replica in active_replicas
            ]
            for replica in active_replicas:
                torch.cuda.synchronize(replica.device)
            inference_started = time.perf_counter()
            start_event.set()
            worker_outputs = [
                future.result() for future in futures
            ]
            for replica in active_replicas:
                torch.cuda.synchronize(replica.device)
            inference_wall_seconds = (
                time.perf_counter() - inference_started
            )
    except (torch.OutOfMemoryError, RuntimeError) as exc:
        is_oom = isinstance(exc, torch.OutOfMemoryError) or (
            "out of memory" in str(exc).lower()
        )
        if not is_oom:
            raise
        raise RuntimeError(
            "Batch-two inference ran out of VRAM. Completed batches remain "
            "atomically checkpointed. Restart or factory-reset the Kaggle "
            "session, then rerun from the top; no sharding or quantization "
            "fallback was attempted."
        ) from exc
    new_attempt_count = sum(
        len(records) for records in worker_outputs
    )

predictions_df = load_resume_checkpoint(EXPECTED_SAMPLE_POSITIONS)
if (
    len(predictions_df) != EXPECTED_DIAGNOSTIC_ROWS
    or predictions_df["content_hash"].duplicated().any()
    or set(predictions_df["content_hash"].astype(str))
    != selected_hashes
):
    raise RuntimeError(
        "Inference is incomplete: exactly 27 unique checkpointed "
        "predictions are required before Gold evaluation."
    )
predictions_df = predictions_df.sort_values(
    "sample_position"
).reset_index(drop=True)
atomic_write_csv(predictions_df, PREDICTIONS_PATH)
INFERENCE_COMPLETED_AT_UTC = utc_now()
print(
    f"Inference complete: resumed={initial_resume_count}, "
    f"new={new_attempt_count}, total={len(predictions_df)}."
)

## 7. Evaluation

Only now are predictions joined to human Gold. Report quality, disagreement direction, score errors, and class inflation without an automatic acceptance threshold. The known 101-image V1 metrics are contextual only; same-27 V1 results are used only when an explicit prediction artifact exists.

In [ ]:
numeric_prediction_columns = [
    "pred_public_relevance",
    "pred_harm_urgency",
    "pred_vulnerability",
    "pred_sensitivity_score",
]
for column in numeric_prediction_columns:
    predictions_df[column] = pd.to_numeric(
        predictions_df[column], errors="coerce"
    )

evaluation = gold_evaluation.merge(
    predictions_df,
    on=["content_hash", "sample_position"],
    how="inner",
    validate="one_to_one",
)
if len(evaluation) != EXPECTED_DIAGNOSTIC_ROWS:
    raise RuntimeError("Gold/prediction join did not contain exactly 27 rows")
valid = evaluation.loc[evaluation["parse_ok"]].copy()
parse_successes = len(valid)
parse_failures = len(evaluation) - parse_successes
if valid.empty:
    raise RuntimeError("No strictly parsed predictions are available")

level_accuracy = accuracy_score(
    valid["sensitivity_level"], valid["pred_sensitivity_level"]
)
macro_f1 = f1_score(
    valid["sensitivity_level"],
    valid["pred_sensitivity_level"],
    labels=list(LEVEL_ORDER),
    average="macro",
    zero_division=0,
)
linear_kappa = cohen_kappa_score(
    valid["sensitivity_level"],
    valid["pred_sensitivity_level"],
    labels=list(LEVEL_ORDER),
    weights="linear",
)
quadratic_kappa = cohen_kappa_score(
    valid["sensitivity_level"],
    valid["pred_sensitivity_level"],
    labels=list(LEVEL_ORDER),
    weights="quadratic",
)
exact_score_accuracy = accuracy_score(
    valid["sensitivity_score"], valid["pred_sensitivity_score"]
)
score_mae = mean_absolute_error(
    valid["sensitivity_score"], valid["pred_sensitivity_score"]
)
component_accuracies = {
    component: accuracy_score(
        valid[component], valid[f"pred_{component}"]
    )
    for component in (
        "public_relevance", "harm_urgency", "vulnerability"
    )
}

gold_high = evaluation["sensitivity_level"] == "high"
valid_gold_high = gold_high & evaluation["parse_ok"]
high_true_positives = int(
    (
        valid_gold_high
        & (evaluation["pred_sensitivity_level"] == "high")
    ).sum()
)
gold_high_count = int(gold_high.sum())
high_recall = high_true_positives / gold_high_count
high_to_medium = int(
    (
        valid_gold_high
        & (evaluation["pred_sensitivity_level"] == "medium")
    ).sum()
)
high_to_low = int(
    (
        valid_gold_high
        & (evaluation["pred_sensitivity_level"] == "low")
    ).sum()
)
high_parse_failures = int((gold_high & ~evaluation["parse_ok"]).sum())

confusion = confusion_matrix(
    valid["sensitivity_level"],
    valid["pred_sensitivity_level"],
    labels=list(LEVEL_ORDER),
)
confusion_frame = pd.DataFrame(
    confusion,
    index=[f"Gold {level.upper()}" for level in LEVEL_ORDER],
    columns=[f"Qwen {level.upper()}" for level in LEVEL_ORDER],
)
human_distribution = (
    evaluation["sensitivity_level"]
    .value_counts()
    .reindex(LEVEL_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)
qwen_distribution = (
    valid["pred_sensitivity_level"]
    .value_counts()
    .reindex(LEVEL_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)
qwen_distribution["parse_failure"] = parse_failures

level_rank = {"low": 0, "medium": 1, "high": 2}
valid["level_error"] = (
    valid["pred_sensitivity_level"].map(level_rank)
    - valid["sensitivity_level"].map(level_rank)
)
direction = {
    "exact_level_agreements": int((valid["level_error"] == 0).sum()),
    "underestimation_count": int((valid["level_error"] < 0).sum()),
    "overestimation_count": int((valid["level_error"] > 0).sum()),
    "parse_failures_not_directionally_classified": parse_failures,
}
valid["score_error"] = (
    valid["pred_sensitivity_score"] - valid["sensitivity_score"]
)
valid["absolute_score_error"] = valid["score_error"].abs()
absolute_score_error_distribution = (
    valid["absolute_score_error"]
    .value_counts()
    .sort_index()
    .astype(int)
    .to_dict()
)

component_disagreements = {
    component: int(
        (valid[component] != valid[f"pred_{component}"]).sum()
    )
    for component in (
        "public_relevance", "harm_urgency", "vulnerability"
    )
}
maximum_component_errors = max(component_disagreements.values())
most_disagreement_dimensions = sorted(
    component
    for component, count in component_disagreements.items()
    if count == maximum_component_errors
)

diagnostic_columns = [
    "content_hash", "blind_id", "public_relevance", "harm_urgency",
    "vulnerability", "sensitivity_score", "sensitivity_level",
    "pred_public_relevance", "pred_harm_urgency", "pred_vulnerability",
    "pred_sensitivity_score", "pred_sensitivity_level",
    "pred_sensitivity_rationale", "pred_annotation_confidence",
    "parse_ok", "parse_error", "raw_model_output",
]
evaluation_diagnostics = evaluation.merge(
    valid[["content_hash", "level_error", "score_error", "absolute_score_error"]],
    on="content_hash",
    how="left",
    validate="one_to_one",
)
component_difference = (
    (evaluation_diagnostics["public_relevance"]
     != evaluation_diagnostics["pred_public_relevance"])
    | (evaluation_diagnostics["harm_urgency"]
       != evaluation_diagnostics["pred_harm_urgency"])
    | (evaluation_diagnostics["vulnerability"]
       != evaluation_diagnostics["pred_vulnerability"])
)
disagreements = evaluation_diagnostics.loc[
    (~evaluation_diagnostics["parse_ok"])
    | component_difference
    | (
        evaluation_diagnostics["sensitivity_level"]
        != evaluation_diagnostics["pred_sensitivity_level"]
    )
].copy()
disagreements = disagreements.sort_values(
    ["parse_ok", "absolute_score_error"],
    ascending=[True, False],
    na_position="first",
)
largest_score_errors = (
    valid.sort_values(
        ["absolute_score_error", "content_hash"],
        ascending=[False, True],
    )[diagnostic_columns + ["score_error", "absolute_score_error"]]
    .head(15)
)

v1_same_27: dict[str, Any] = {
    "available": False,
    "reason": "No explicit V1 prediction artifact was available.",
}
v1_path = next(
    (path for path in V1_PREDICTION_CANDIDATES if path.is_file()),
    None,
)
if v1_path is not None:
    v1_predictions = pd.read_csv(v1_path, keep_default_na=False)
    if (
        "content_hash" not in v1_predictions.columns
        or v1_predictions["content_hash"].duplicated().any()
    ):
        raise RuntimeError(
            f"explicit V1 artifact is incompatible: {v1_path}"
        )
    v1_selected = v1_predictions[
        v1_predictions["content_hash"].astype(str).isin(selected_hashes)
    ].copy()
    if (
        len(v1_selected) == EXPECTED_DIAGNOSTIC_ROWS
        and set(v1_selected["content_hash"].astype(str)) == selected_hashes
    ):
        v1_selected["parse_ok"] = v1_selected["parse_ok"].map(
            lambda value: str(value).lower() == "true"
        )
        v1_join = gold_evaluation.merge(
            v1_selected,
            on="content_hash",
            how="inner",
            validate="one_to_one",
        )
        v1_valid = v1_join[v1_join["parse_ok"]]
        v1_same_27 = {
            "available": True,
            "artifact_path": str(v1_path),
            "rows": len(v1_join),
            "parse_successes": len(v1_valid),
            "sensitivity_level_accuracy": accuracy_score(
                v1_valid["sensitivity_level"],
                v1_valid["pred_sensitivity_level"],
            ),
            "public_relevance_accuracy": accuracy_score(
                v1_valid["public_relevance"],
                pd.to_numeric(
                    v1_valid["pred_public_relevance"], errors="raise"
                ),
            ),
            "qwen_distribution": (
                v1_valid["pred_sensitivity_level"]
                .value_counts()
                .reindex(LEVEL_ORDER, fill_value=0)
                .astype(int)
                .to_dict()
            ),
            "comparison_scope": (
                "Direct same-27 descriptive comparison only; no significance "
                "claim or automatic acceptance decision."
            ),
        }
    else:
        v1_same_27 = {
            "available": False,
            "artifact_path": str(v1_path),
            "reason": "Explicit V1 artifact does not contain all 27 selected hashes.",
        }

metrics_payload = {
    "diagnostic_rows": len(evaluation),
    "strictly_parsed_rows_used_for_quality_metrics": len(valid),
    "parse_success_rate": parse_successes / len(evaluation),
    "parse_successes": parse_successes,
    "parse_failures": parse_failures,
    "sensitivity_level": {
        "accuracy": level_accuracy,
        "macro_f1": macro_f1,
        "linear_weighted_cohen_kappa": linear_kappa,
        "quadratic_weighted_cohen_kappa": quadratic_kappa,
    },
    "sensitivity_score": {
        "exact_accuracy": exact_score_accuracy,
        "mae": score_mae,
        "absolute_error_distribution": absolute_score_error_distribution,
    },
    "component_accuracy": component_accuracies,
    "component_disagreement_counts": component_disagreements,
    "most_disagreement_dimensions": most_disagreement_dimensions,
    "high_performance": {
        "gold_high_cases": gold_high_count,
        "recall": high_recall,
        "high_to_medium": high_to_medium,
        "high_to_low": high_to_low,
        "high_parse_failures": high_parse_failures,
    },
    "confusion_matrix": {
        "labels": list(LEVEL_ORDER),
        "rows_gold_columns_qwen": confusion.tolist(),
        "parse_failures_excluded": parse_failures,
    },
    "human_gold_distribution": human_distribution,
    "qwen_v2_distribution": qwen_distribution,
    "direction_of_level_disagreement": direction,
    "v1_101_contextual_reference": V1_REFERENCE_101,
    "v1_same_27": v1_same_27,
    "interpretation_note": (
        "Evidence only. No pass/fail threshold is applied; inspect whether "
        "changes reflect rubric alignment rather than indiscriminate inflation."
    ),
}
run_metadata = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "prompt_text": PROMPT_V2,
    "prompt_sha256": PROMPT_V2_SHA256,
    "seed": SEED,
    "model_layout": "one_complete_independent_fp16_replica_per_gpu",
    "replica_device_maps": REPLICA_DEVICE_MAPS,
    "self_diagnostic_preflight": PREFLIGHT_REPORT,
    "model_loading_skipped_for_complete_resume": not bool(inference_items),
    "gpu_names": GPU_NAMES,
    "gpu_count": GPU_COUNT,
    "dtype": "torch.float16",
    "quantized": False,
    "batch_size_per_gpu": BATCH_SIZE_PER_GPU,
    "true_concurrent_inference": True,
    "generation_parameters": GENERATION_PARAMETERS,
    "image_preprocessing": {
        "in_memory_only": True,
        "preserve_aspect_ratio": True,
        "maximum_image_side_pixels": MAX_IMAGE_SIDE,
        "resampling": "PIL.Image.Resampling.LANCZOS",
        "exif_transpose": True,
        "color_mode": "RGB",
    },
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "huggingface_hub_version": huggingface_hub.__version__,
    "sklearn_version": sklearn.__version__,
    "cuda_version": torch.version.cuda,
    "initial_resumed_predictions": initial_resume_count,
    "new_inference_attempts": new_attempt_count,
    "completed_predictions": len(predictions_df),
    "inference_wall_seconds_this_session": inference_wall_seconds,
    "inference_completed_at_utc": INFERENCE_COMPLETED_AT_UTC,
    "total_notebook_runtime_seconds": time.perf_counter() - NOTEBOOK_STARTED_PERF,
    "resume_signature": RESUME_SIGNATURE,
    "artifacts": {
        "predictions": str(PREDICTIONS_PATH),
        "metrics": str(METRICS_PATH),
        "disagreements": str(DISAGREEMENTS_PATH),
        "run_metadata": str(RUN_METADATA_PATH),
    },
}

atomic_write_csv(disagreements, DISAGREEMENTS_PATH)
atomic_write_json(metrics_payload, METRICS_PATH)
atomic_write_json(run_metadata, RUN_METADATA_PATH)

display(pd.DataFrame([{
    "parse_success_rate": metrics_payload["parse_success_rate"],
    "level_accuracy": level_accuracy,
    "macro_f1": macro_f1,
    "linear_kappa": linear_kappa,
    "quadratic_kappa": quadratic_kappa,
    "exact_score_accuracy": exact_score_accuracy,
    "score_mae": score_mae,
    "high_recall": high_recall,
}]))
display(confusion_frame)
display(pd.DataFrame({
    "Human Gold": pd.Series(human_distribution),
    "Qwen V2": pd.Series(qwen_distribution),
}).fillna(0).astype(int))
display(pd.Series(direction, name="count").to_frame())
display(pd.Series(component_disagreements, name="disagreements").to_frame())
display(largest_score_errors)
display(disagreements)
print(
    "Prompt V2 diagnostic artifacts saved. Interpret manually; "
    "no acceptance threshold was applied."
)